# Repelón Risk Lens — V4 · 1/2: Riesgo territorial (CHIRPS)

**Ejecutar de arriba abajo.** Produce el cribado de 1.848 predios, con CHIRPS como única fuente de lluvia del índice principal. No requiere GPM ni NASA POWER.

**Interpretación:** modelo exploratorio, sin validación en campo. Las puntuaciones no prueban pérdidas agrícolas. El notebook 2 es independiente y contiene las comparaciones y el escenario de yuca.


## 0. Configuración (aquí se cambian los parámetros; el resto no se toca)

In [ ]:
PROJECT_ID = 'agro-513620'
RUTA_BASE = '/content/drive/MyDrive/Agro'
RUTA_PREDIOS = f'{RUTA_BASE}/repelon_predios_rurales.gpkg'

# Fecha final del análisis. None = hoy. Para reproducir resultados escribe 'AAAA-MM-DD'.
FECHA_FIN = '2026-11-15'   # fijo para reproducir (None = hoy)
VENTANA_DIAS = 60          # días hacia atrás que se analizan
NUBOSIDAD_MAX = 30         # % máximo de nubes por escena Sentinel-2
ESCALA_S2 = 10             # metros por píxel al extraer índices (10 m = más predios con dato)
ESCALA_CHIRPS = 5566       # metros (CHIRPS, 0.05 grado)
CHIRPS_ID = 'UCSB-CHC/CHIRPS/V3/PENTAD'
# Periodo de lluvia: SOLO pentadas completas realmente disponibles. No se extrapola a 60 días.
LLUVIA_INI_STR = '2026-09-16'   # primer día de la primera pentada
LLUVIA_FIN_STR = '2026-10-31'   # último día incluido: la última pentada de octubre tiene 6 días
N_PENTADAS = 9                  # 16-20,21-25,26-30 sep + 1-5,6-10,11-15,16-20,21-25,26-31 oct
ANIOS_CLIMA = 10           # años anteriores (2016-2025) para la lluvia "normal"
TAMANO_LOTE = 500          # predios por consulta a Earth Engine

# Pesos de la vulnerabilidad (deben sumar 1)
PESOS_V = {'NDVI': 0.4, 'NDMI': 0.4, 'SAVI': 0.2}

# Umbrales (crítico, sano). SON SUPUESTOS INICIALES: hay que calibrarlos con datos locales.
# Si el índice vale <= crítico -> estrés 1. Si vale >= sano -> estrés 0. Entre ambos, lineal.
UMBRALES = {
    'NDVI': (0.2, 0.7),
    'NDMI': (-0.1, 0.3),
    'SAVI': (0.05, 0.5),
}

PESO_AMENAZA = 0.5         # Riesgo = PESO_AMENAZA * A + (1 - PESO_AMENAZA) * V

## 1. Librerías, autenticación y Drive

In [ ]:
import warnings
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import ee
import geemap
import shutil
from google.colab import drive

warnings.filterwarnings('ignore', category=UserWarning)

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
print('✅ Conectado a Earth Engine')

drive.mount('/content/drive')
print('✅ Drive montado')

## 2. Carga y limpieza de predios (búfer adaptativo)

In [ ]:
# 1. Cargar GPKG desde Google Drive


ruta_drive = "/content/drive/MyDrive/Agro/repelon_predios_rurales.gpkg"
ruta_local = "/content/repelon_predios_rurales.gpkg"

shutil.copy2(ruta_drive, ruta_local)

gdf = gpd.read_file(ruta_local)

print("Registros originales:", len(gdf))


# 2. Eliminar duplicados con geometrías idénticas
gdf_limpio = gdf.drop_duplicates(
    subset=["CODIGO", "geometry"]
).copy()


# 3. Para códigos con varias geometrías, conservar la de mayor área
gdf_limpio["area_m2"] = gdf_limpio.geometry.area

gdf_limpio = (
    gdf_limpio
    .sort_values("area_m2", ascending=False)
    .drop_duplicates(subset="CODIGO", keep="first")
    .drop(columns="area_m2")
    .reset_index(drop=True)
)


# 4. Verificaciones
print("Registros finales:", len(gdf_limpio))
print("Códigos únicos:", gdf_limpio["CODIGO"].nunique())
print("Códigos repetidos:", gdf_limpio["CODIGO"].duplicated().sum())

print(
    "Superficie total (ha):",
    round(gdf_limpio.geometry.area.sum() / 10000, 3)
)

assert len(gdf_limpio) == 1848
assert gdf_limpio["CODIGO"].is_unique

# Usar la capa limpia como base para el resto del proyecto
gdf = gdf_limpio.copy()

print("GeoDataFrame de trabajo:", len(gdf))
print("Códigos únicos:", gdf["CODIGO"].nunique())

## 3. Fechas y función auxiliar de extracción a Earth Engine

In [ ]:
FIN = pd.Timestamp(FECHA_FIN) if FECHA_FIN else pd.Timestamp.now().normalize()
INICIO = FIN - pd.Timedelta(days=VENTANA_DIAS)


def f(fecha):
    return fecha.strftime('%Y-%m-%d')


print(f'Ventana Sentinel-2: {f(INICIO)} a {f(FIN)}')

# Rectángulo que cubre todos los predios (en lat/lon), usado para filtrar imágenes
x0, y0, x1, y1 = gdf.to_crs(4326).total_bounds
region = ee.Geometry.Rectangle([float(x0), float(y0), float(x1), float(y1)])


def extraer_por_lotes(imagen, escala):
    """Promedia 'imagen' dentro de cada predio. Devuelve DataFrame con CODIGO + valores."""
    base = gdf[['CODIGO', 'geometry']]
    resultados = []
    for i in range(0, len(base), TAMANO_LOTE):
        lote = base.iloc[i:i + TAMANO_LOTE].to_crs(4326)   # Earth Engine usa lat/lon
        fc = geemap.geopandas_to_ee(lote)
        salida = imagen.reduceRegions(
            collection=fc, reducer=ee.Reducer.mean(), scale=escala, tileScale=4
        )
        info = salida.getInfo()
        resultados.append(pd.DataFrame([x['properties'] for x in info['features']]))
        print(f'  lote {i // TAMANO_LOTE + 1} de {-(-len(base) // TAMANO_LOTE)} listo')
    df = pd.concat(resultados, ignore_index=True)
    df['CODIGO'] = df['CODIGO'].astype(str).str.strip()
    return df.drop_duplicates('CODIGO')

## 4. Vulnerabilidad (Sentinel-2)

In [ ]:
def enmascarar_s2(img):
    """Quita nubes, sombras y píxeles dudosos usando la banda SCL (4=vegetación, 5=suelo)."""
    scl = img.select('SCL')
    return img.updateMask(scl.eq(4).Or(scl.eq(5)))


s2_col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(region)
    .filterDate(f(INICIO), f(FIN))
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', NUBOSIDAD_MAX))
)
n_escenas = s2_col.size().getInfo()
print(f'Escenas Sentinel-2 disponibles: {n_escenas}')
assert n_escenas > 0, 'No hay escenas en la ventana: amplía VENTANA_DIAS o NUBOSIDAD_MAX'

# Mediana de las escenas ya enmascaradas; se divide entre 10000 para pasar a reflectancia (0-1)
s2 = s2_col.map(enmascarar_s2).select(['B4', 'B8', 'B11']).median().divide(10000)

nir, red = s2.select('B8'), s2.select('B4')
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndmi = s2.normalizedDifference(['B8', 'B11']).rename('NDMI')
savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
indices = ee.Image.cat([ndvi, ndmi, savi])

print('Extrayendo índices por predio...')
df_s2 = extraer_por_lotes(indices, ESCALA_S2)
df_s2.to_csv(f'{RUTA_BASE}/repel_indices_s2.csv', index=False)

for col in ['NDVI', 'NDMI', 'SAVI']:
    if col not in df_s2.columns:
        df_s2[col] = float('nan')

gdf = gdf.drop(columns=['NDVI', 'NDMI', 'SAVI'], errors='ignore')
gdf = gdf.merge(df_s2[['CODIGO', 'NDVI', 'NDMI', 'SAVI']], on='CODIGO', how='left')
sin_dato = gdf['NDVI'].isna().sum()
print(f'Predios sin dato Sentinel-2: {sin_dato} de {len(gdf)} ({100 * sin_dato / len(gdf):.1f}%)')


def estres(serie, critico, sano):
    """0 = sano, 1 = crítico. Lineal entre los dos umbrales."""
    return ((sano - serie) / (sano - critico)).clip(0, 1)


gdf['V_Vulnerability'] = sum(
    PESOS_V[k] * estres(gdf[k], *UMBRALES[k]) for k in PESOS_V
)   # si falta un índice, V queda vacío (NaN): no se inventa el dato

print('\n--- TOP 10 PREDIOS CON MAYOR VULNERABILIDAD ---')
print(
    gdf[['CODIGO', 'NDVI', 'NDMI', 'SAVI', 'V_Vulnerability']]
    .sort_values('V_Vulnerability', ascending=False)
    .head(10)
)

## 5. Amenaza (CHIRPS V3 pentadal): lluvia del periodo frente a lo normal

In [ ]:
LLUVIA_INI = pd.Timestamp(LLUVIA_INI_STR)
LLUVIA_FIN = pd.Timestamp(LLUVIA_FIN_STR)
UN_DIA = pd.Timedelta(days=1)


def pentadas(ini, fin):
    """Pentadas CHIRPS cuyo inicio cae entre ini y fin (ambos incluidos). La última pentada de meses de 31 días dura 6 días."""
    col = (ee.ImageCollection(CHIRPS_ID)
           .filterDate(f(ini), f(fin + UN_DIA)).select('precipitation'))
    ms = col.aggregate_array('system:time_start').getInfo()
    return col, pd.to_datetime(ms, unit='ms')


# 5.1 Periodo actual: se verifica que haya exactamente las pentadas esperadas
col_act, fechas_act = pentadas(LLUVIA_INI, LLUVIA_FIN)
firma_act = [d.strftime('%m-%d') for d in fechas_act]
print(f'Pentadas actuales: {len(fechas_act)} -> {firma_act}')
assert len(fechas_act) == N_PENTADAS, (
    f'Se esperaban {N_PENTADAS} pentadas y hay {len(fechas_act)}. No se rellena ni se extrapola: ajusta el periodo.')
dias_periodo = (LLUVIA_FIN - LLUVIA_INI).days + 1
print(f'Periodo de lluvia: {f(LLUVIA_INI)} a {f(LLUVIA_FIN)} ({dias_periodo} días)')
print('Propiedades de la primera imagen (revisa si indican producto preliminar/final):',
      col_act.first().propertyNames().getInfo())
print('Resolución nativa (m):', col_act.first().projection().nominalScale().getInfo())
precip_actual = col_act.sum().rename('precip_mm')

# 5.2 Lluvia normal: MISMAS pentadas (mismas fechas de inicio) en los 10 años anteriores
sumas, anios_usados = [], []
for k in range(1, ANIOS_CLIMA + 1):
    col_k, fechas_k = pentadas(LLUVIA_INI - pd.DateOffset(years=k), LLUVIA_FIN - pd.DateOffset(years=k))
    firma_k = [d.strftime('%m-%d') for d in fechas_k]
    if firma_k == firma_act:
        sumas.append(col_k.sum())
        anios_usados.append(int(fechas_k[0].year))
        print(f'  {fechas_k[0].year}: {len(fechas_k)} pentadas, mismas fechas ✓')
    else:
        print(f'  ⚠️ {LLUVIA_INI.year - k}: pentadas distintas {firma_k}. Año omitido.')
assert len(sumas) >= 8, 'Muy pocos años completos para una normal de 10 años'
ANIOS_USADOS = anios_usados
precip_normal = ee.ImageCollection.fromImages(sumas).mean().rename('precip_normal_mm')   # normal por píxel
print(f'Años usados para la normal ({len(ANIOS_USADOS)}): {ANIOS_USADOS}')

# 5.3 Promedios de la zona (solo informativos y para el JSON de parámetros)
reg = precip_actual.addBands(precip_normal).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=region, scale=ESCALA_CHIRPS, maxPixels=1e9).getInfo()
LLUVIA_ACTUAL_REGIONAL_MM = reg['precip_mm']
LLUVIA_NORMAL_REGIONAL_MM = reg['precip_normal_mm']
print(f'Zona: lluvia actual {LLUVIA_ACTUAL_REGIONAL_MM:.1f} mm | normal {LLUVIA_NORMAL_REGIONAL_MM:.1f} mm')

# 5.4 Valor del píxel que contiene el centroide de cada predio (actual y normal del MISMO píxel)
cent = gdf[['CODIGO', 'geometry']].copy()
cent['geometry'] = cent.geometry.centroid
cent = cent.to_crs(4326)
img_ll = precip_actual.addBands(precip_normal)
filas = []
for i in range(0, len(cent), TAMANO_LOTE):
    fc = geemap.geopandas_to_ee(cent.iloc[i:i + TAMANO_LOTE])
    out = img_ll.sampleRegions(collection=fc, properties=['CODIGO'], scale=ESCALA_CHIRPS)
    filas += [x['properties'] for x in out.getInfo()['features']]
df_chirps = pd.DataFrame(filas)
df_chirps['CODIGO'] = df_chirps['CODIGO'].astype(str).str.strip()
df_chirps = df_chirps.drop_duplicates('CODIGO')
print(f'Predios con dato de lluvia CHIRPS: {len(df_chirps)} de {len(gdf)}')
assert len(df_chirps) == len(gdf), 'Faltan predios sin dato CHIRPS: revisa antes de seguir'
df_chirps.to_csv(f'{RUTA_BASE}/repel_amenaza_chirps.csv', index=False)

# 5.5 Amenaza con la fórmula original: 0 si llovió lo normal o más; 1 si no llovió nada
gdf = gdf.drop(columns=['precip_mm', 'precip_normal_mm', 'Precip_pct_normal', 'Amenaza_Norm'], errors='ignore')
gdf = gdf.merge(df_chirps[['CODIGO', 'precip_mm', 'precip_normal_mm']], on='CODIGO', how='left')
assert (gdf['precip_normal_mm'] > 0).all(), 'Normal en cero o vacía en algún predio'
gdf['Precip_pct_normal'] = 100 * gdf['precip_mm'] / gdf['precip_normal_mm']
gdf['Amenaza_Norm'] = (1 - gdf['precip_mm'] / gdf['precip_normal_mm']).clip(0, 1)
gdf['fuente_lluvia'] = 'CHIRPS V3 pentadal'

print(gdf[['precip_mm', 'precip_normal_mm', 'Precip_pct_normal', 'Amenaza_Norm']].describe().round(3))
print('Valores distintos de amenaza:', gdf['Amenaza_Norm'].nunique())
print('\nNota: píxel de ~5,6 km: más variación que GPM, pero sigue siendo un producto satelital sin validar en campo.')

## 6. Riesgo final

In [ ]:
gdf['Riesgo_Index'] = (
    PESO_AMENAZA * gdf['Amenaza_Norm'] + (1 - PESO_AMENAZA) * gdf['V_Vulnerability']
)


def clasificar_riesgo(score):
    if pd.isna(score):
        return 'Sin dato'
    if score > 0.75:
        return 'Crítico'
    if score > 0.50:
        return 'Alto'
    if score > 0.25:
        return 'Moderado'
    return 'Bajo'


gdf['Nivel_Riesgo'] = gdf['Riesgo_Index'].apply(clasificar_riesgo)

gdf.to_file(f'{RUTA_BASE}/repel_riesgo_consolidado_chirps.gpkg', driver='GPKG')   # para QGIS (versión CHIRPS)
pd.DataFrame(gdf.drop(columns='geometry')).to_csv(
    f'{RUTA_BASE}/repel_riesgo_consolidado_chirps.csv', index=False
)

print('✅ Riesgo calculado y guardado en Drive')
print(gdf['Nivel_Riesgo'].value_counts())

## 6b. Validaciones obligatorias del modelo (integridad, lluvia, riesgo)

In [ ]:
# --- Integridad ---
assert len(gdf) == 1848, f'Se esperaban 1.848 predios y hay {len(gdf)}'
assert gdf['CODIGO'].is_unique, 'Hay códigos repetidos'
for c in ['Amenaza_Norm', 'V_Vulnerability', 'Riesgo_Index']:
    n_nan = int(gdf[c].isna().sum())
    print(f'Faltantes en {c}: {n_nan}')
    assert n_nan == 0, f'Hay valores faltantes en {c}'
assert not gdf['Nivel_Riesgo'].eq('Sin dato').any()
print('✅ 1.848 predios únicos, sin faltantes en amenaza, vulnerabilidad ni riesgo')

# --- Lluvia ---
print(f'\nPentadas: {N_PENTADAS} ({firma_act}); periodo {f(LLUVIA_INI)} a {f(LLUVIA_FIN)} = {dias_periodo} días')
print(f'Años de la normal: {ANIOS_USADOS}')
print(f'Resolución CHIRPS ~{ESCALA_CHIRPS} m, EPSG:4326 (0,05°)')
print(f'Zona: actual {LLUVIA_ACTUAL_REGIONAL_MM:.1f} mm | normal {LLUVIA_NORMAL_REGIONAL_MM:.1f} mm')

# --- Modelo ---
print('\nDistribución de V, A y riesgo:')
print(gdf[['V_Vulnerability', 'Amenaza_Norm', 'Riesgo_Index']].describe().round(3))
print('\nPredios por nivel:'); print(gdf['Nivel_Riesgo'].value_counts())
print('\nCoherencia de umbrales (riesgo por nivel, mín–máx):')
print(gdf.groupby('Nivel_Riesgo')['Riesgo_Index'].agg(['count', 'min', 'max']).round(3))
print('\nAdvertencia: el número de predios en Alto depende de la fuente de lluvia y de los cortes (supuestos); compara con GPM en la sección 15.')

## 7. Revisión visual (NO es validación)

Estos gráficos solo comprueban que los números se ven razonables.
Validar de verdad = comparar contra datos independientes (ver GUIA_Y_BITACORA.md).

In [ ]:
sns.set_theme(style='whitegrid')
fig, ejes = plt.subplots(1, 3, figsize=(18, 4.5))

sns.histplot(gdf['V_Vulnerability'].dropna(), bins=30, kde=True, ax=ejes[0], color='forestgreen')
ejes[0].set_title('Vulnerabilidad (V) por predio')

sns.histplot(gdf['Precip_pct_normal'].dropna(), bins=30, ax=ejes[1], color='dodgerblue')
ejes[1].set_title('Lluvia del periodo (CHIRPS) como % de lo normal')

gdf['Nivel_Riesgo'].value_counts().reindex(
    ['Crítico', 'Alto', 'Moderado', 'Bajo', 'Sin dato']
).fillna(0).plot(kind='bar', ax=ejes[2], color='darkorange')
ejes[2].set_title('Predios por nivel de riesgo')

plt.tight_layout()
plt.savefig(f'{RUTA_BASE}/revision_visual.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Predios dentro o fuera del distrito de riego, y candidatos para el ejemplo

Usa el cruce hecho en QGIS entre los predios y el polígono oficial del distrito de riego de Repelón (servicio de Distritos de Adecuación de Tierras de la UPRA y la ANT). El archivo `predios_en_distrito.csv` debe estar en la carpeta Agro de Drive y traer una columna `CODIGO`.
Si el archivo no está, usa como respaldo la distancia al punto de referencia del registro de la ADR, que es solo un indicio.
Estar fuera del distrito significa que el predio probablemente NO recibe riego del distrito; no prueba que sea de secano (puede haber bombeo propio u otras fuentes).
Lista predios candidatos fuera del distrito y de más de 1 ha, con riesgo alto y con riesgo bajo, para elegir el predio de ejemplo.

In [ ]:
import os
from shapely.geometry import Point

# Cruce espacial hecho en QGIS: predios que intersecan el polígono oficial del distrito de riego.
# Sube el CSV a la carpeta Agro de Drive.
RUTA_PREDIOS_DISTRITO = f'{RUTA_BASE}/predios_en_distrito_nuevo.csv'
AREA_MIN_HA = 1            # predios de más de 1 ha dan píxeles más limpios

# Distancia al PUNTO de referencia ADR, no al borde del polígono del distrito.
# Punto de referencia del registro de la ADR (solo informativo): 10°29.803' N, 75°7.960' W
LAT_DISTRITO = 10 + 29.803 / 60
LON_DISTRITO = -(75 + 7.960 / 60)
# Calcular distancias en metros con ambas geometrías en el MISMO CRS proyectado.
CRS_DISTANCIA = 'EPSG:32618'
assert gdf.crs is not None, 'Los predios no tienen CRS declarado; revisar el GPKG.'
punto_distrito = gpd.GeoSeries([Point(LON_DISTRITO, LAT_DISTRITO)], crs='EPSG:4326').to_crs(CRS_DISTANCIA).iloc[0]
predios_m = gdf.to_crs(CRS_DISTANCIA)

gdf['area_ha'] = (predios_m.geometry.area / 10000).round(2)     # área del predio ya con el búfer
gdf['dist_distrito_km'] = (predios_m.geometry.centroid.distance(punto_distrito) / 1000).round(2)
assert gdf['dist_distrito_km'].between(0, 100).all(), 'Distancias fuera de rango: revisar CRS y coordenadas'
print('Distancias al punto ADR (km):', gdf['dist_distrito_km'].describe())

if os.path.exists(RUTA_PREDIOS_DISTRITO):
    tabla_d = pd.read_csv(RUTA_PREDIOS_DISTRITO, dtype=str)
    col_cod = next((c for c in tabla_d.columns if c.strip().upper() == 'CODIGO'), None)
    assert col_cod is not None, f'El CSV no tiene una columna CODIGO. Columnas: {list(tabla_d.columns)}'
    # El export de QGIS trae también predios SIN distrito (columna 'asoc' vacía): solo cuentan los que la traen llena
    if 'asoc' in tabla_d.columns:
        tabla_d = tabla_d[tabla_d['asoc'].notna() & (tabla_d['asoc'].str.strip() != '')]
    codigos_distrito = set(tabla_d[col_cod].astype(str).str.strip())
    gdf['en_distrito'] = gdf['CODIGO'].isin(codigos_distrito)
    n_dentro = int(gdf['en_distrito'].sum())
    print(f'Predios marcados dentro del distrito: {n_dentro} de {len(gdf)} (en QGIS, regla centroide dentro: 472)')
    print(
        f"Área de esos predios: {gdf.loc[gdf['en_distrito'], 'area_ha'].sum():.0f} ha "
        '(referencia ADR: área bruta del distrito 6.825 ha, neta 3.613 ha; los predios no están todos irrigados)'
    )
    if n_dentro == 0:
        print('⚠️ Ninguno coincide: revisa que los CODIGO del CSV tengan el mismo formato que los del notebook.')
else:
    print(f'⚠️ No encontré {RUTA_PREDIOS_DISTRITO}. Uso la distancia al punto del distrito como indicio (6 km).')
    gdf['en_distrito'] = gdf['dist_distrito_km'] < 6

cand = gdf[(~gdf['en_distrito']) & (gdf['area_ha'] >= AREA_MIN_HA) & gdf['Riesgo_Index'].notna()]
cols = ['CODIGO', 'area_ha', 'dist_distrito_km', 'NDVI', 'NDMI', 'V_Vulnerability', 'Riesgo_Index', 'Nivel_Riesgo']
print(f'\nPredios candidatos fuera del distrito y de más de {AREA_MIN_HA} ha: {len(cand)}')
if len(cand) == 0:
    print('Ninguno: baja AREA_MIN_HA y vuelve a correr esta celda.')
else:
    print('\n--- Riesgo ALTO ---')
    print(cand.sort_values('Riesgo_Index', ascending=False)[cols].head(8))
    print('\n--- Riesgo BAJO ---')
    print(cand.sort_values('Riesgo_Index')[cols].head(8))

## 9b. Cobertura del suelo (ESA WorldCover): ¿el predio es cultivo, pasto, monte o construcción?

Los índices (NDVI, NDMI, SAVI) miden verdor y humedad de cualquier cobertura. Esta sección le pone a cada predio una etiqueta de qué hay dentro, usando el mapa de cobertura del suelo ESA WorldCover 2021 (10 m). **No borra predios: solo los marca**, para saber cuáles son agrícolas (cultivo o pastizal) y cuáles no (monte, construcción, suelo desnudo, agua).

In [ ]:
# Cobertura del suelo por predio (ESA WorldCover 2021, 10 m)
CLASES_WC = {10: 'Arboles', 20: 'Matorral', 30: 'Pastizal', 40: 'Cultivo', 50: 'Construido',
             60: 'Suelo_desnudo', 70: 'Nieve', 80: 'Agua', 90: 'Humedal', 95: 'Manglar', 100: 'Musgo'}
UMBRAL_AGRO = 0.5     # un predio es "agropecuario" si Cultivo + Pastizal cubren al menos esta fracción

wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

filas = []
base = gdf[['CODIGO', 'geometry']]
for i in range(0, len(base), TAMANO_LOTE):
    lote = base.iloc[i:i + TAMANO_LOTE].to_crs(4326)
    fc = geemap.geopandas_to_ee(lote)
    salida = wc.reduceRegions(
        collection=fc, reducer=ee.Reducer.frequencyHistogram(), scale=10, tileScale=4
    ).getInfo()
    for x in salida['features']:
        p = x['properties']
        h = p.get('histogram') or p.get('Map') or {}
        filas.append({'CODIGO': str(p['CODIGO']).strip(), 'hist': h})
    print(f'  lote {i // TAMANO_LOTE + 1} de {-(-len(base) // TAMANO_LOTE)} listo')


def resumir(h):
    """Convierte el conteo de píxeles por clase en: clase dominante y fracción agropecuaria."""
    if not h:
        return pd.Series({'cobertura_dominante': 'Sin dato', 'pct_agropecuario': float('nan')})
    total = sum(h.values())
    por_clase = {CLASES_WC.get(int(float(k)), str(k)): v / total for k, v in h.items()}
    dominante = max(por_clase, key=por_clase.get)
    agro = por_clase.get('Cultivo', 0) + por_clase.get('Pastizal', 0)
    return pd.Series({'cobertura_dominante': dominante, 'pct_agropecuario': round(100 * agro, 1)})


df_wc = pd.DataFrame(filas).drop_duplicates('CODIGO')
df_wc = pd.concat([df_wc[['CODIGO']], df_wc['hist'].apply(resumir)], axis=1)
df_wc['uso_agropecuario'] = df_wc['pct_agropecuario'] >= 100 * UMBRAL_AGRO
df_wc.to_csv(f'{RUTA_BASE}/repel_cobertura.csv', index=False)

gdf = gdf.drop(columns=['cobertura_dominante', 'pct_agropecuario', 'uso_agropecuario'], errors='ignore')
gdf = gdf.merge(df_wc, on='CODIGO', how='left')
gdf['uso_agropecuario'] = gdf['uso_agropecuario'].fillna(False).astype(bool)

print('\n--- Cobertura dominante de los predios ---')
print(gdf['cobertura_dominante'].value_counts(dropna=False))
print(f"\nPredios agropecuarios (cultivo + pastizal >= {int(100 * UMBRAL_AGRO)} %): "
      f"{int(gdf['uso_agropecuario'].sum())} de {len(gdf)}")

print('\n--- Nivel de riesgo según uso agropecuario ---')
print(pd.crosstab(gdf['uso_agropecuario'], gdf['Nivel_Riesgo']))

print('\n--- Los 10 predios de MENOR vulnerabilidad (>= 1 ha): ¿son agrícolas? ---')
print(gdf[gdf['area_ha'] >= 1].nsmallest(10, 'V_Vulnerability')
      [['CODIGO', 'area_ha', 'V_Vulnerability', 'cobertura_dominante', 'pct_agropecuario']])
print('\n--- Los 10 predios de MAYOR vulnerabilidad (>= 1 ha) ---')
print(gdf[gdf['area_ha'] >= 1].nlargest(10, 'V_Vulnerability')
      [['CODIGO', 'area_ha', 'V_Vulnerability', 'cobertura_dominante', 'pct_agropecuario']])

9c. Propiedades físicas del suelo (SoilGrids

In [ ]:
# --- Celda corregida para la Sección 9c ---
# SoilGrids clay_mean (g/kg): convertir a porcentaje dividiendo por 10.
# Extracción de arcilla superficial (0-5 cm) desde SoilGrids
soil_img = ee.Image('projects/soilgrids-isric/clay_mean').select('clay_0-5cm_mean')

filas_suelo = []
base = gdf[['CODIGO', 'geometry']]
for i in range(0, len(base), TAMANO_LOTE):
    lote = base.iloc[i:i + TAMANO_LOTE].to_crs(4326)
    fc = geemap.geopandas_to_ee(lote)
    salida = soil_img.reduceRegions(
        collection=fc, reducer=ee.Reducer.mean(), scale=250, tileScale=4
    )
    info = salida.getInfo()
    filas_suelo.append(pd.DataFrame([x['properties'] for x in info['features']]))
    print(f'  lote suelo {i // TAMANO_LOTE + 1} de {-(-len(base) // TAMANO_LOTE)} listo')

df_suelo = pd.concat(filas_suelo, ignore_index=True)
df_suelo['CODIGO'] = df_suelo['CODIGO'].astype(str).str.strip()
df_suelo = df_suelo.drop_duplicates('CODIGO')

# Integrarlo al GeoDataFrame principal
gdf = gdf.drop(columns=['soil_clay_pct'], errors='ignore')
gdf = gdf.merge(df_suelo[['CODIGO', 'mean']], on='CODIGO', how='left').rename(columns={'mean': 'soil_clay_pct'})
gdf['soil_clay_pct'] = pd.to_numeric(gdf['soil_clay_pct'], errors='coerce') / 10.0
assert gdf['soil_clay_pct'].dropna().between(0, 100).all(), 'Porcentajes de arcilla fuera de rango: revisar unidades de SoilGrids'
print('Arcilla superficial (%):', gdf['soil_clay_pct'].describe())

print(f"✅ Datos de suelo añadidos. Predios con dato de arcilla: {gdf['soil_clay_pct'].notna().sum()} de {len(gdf)}")

In [ ]:
# Exportación final con contexto territorial corregido (sin modificar riesgo CHIRPS)
import shutil
ruta_gpkg_final = f'{RUTA_BASE}/repel_riesgo_consolidado_chirps_contexto_corregido.gpkg'
ruta_csv_final = f'{RUTA_BASE}/repel_riesgo_consolidado_chirps_contexto_corregido.csv'
ruta_gpkg_local = '/content/repel_riesgo_consolidado_chirps_contexto_corregido.gpkg'
ruta_csv_local = '/content/repel_riesgo_consolidado_chirps_contexto_corregido.csv'
gdf.to_file(ruta_gpkg_local, driver='GPKG')
gdf.drop(columns='geometry').to_csv(ruta_csv_local, index=False)
shutil.copy2(ruta_gpkg_local, ruta_gpkg_final)
shutil.copy2(ruta_csv_local, ruta_csv_final)
print('Exportaciones de contexto corregido guardadas en Drive:')
print(ruta_gpkg_final)
print(ruta_csv_final)
print(gdf['Nivel_Riesgo'].value_counts())


## 10. NDVI mensual del predio de ejemplo

Escribe abajo el CODIGO del predio elegido en la sección 9 (entre comillas) y corre la celda.
Calcula la curva de verdor mes a mes de ese predio (2021 a 2025) y marca los meses en que el NDVI baja del umbral supuesto: son los meses de posible campo desnudo o estrés.
Un NDVI bajo es solo una pista: puede ser suelo desnudo, barbecho o estrés, y el índice no los distingue.

In [ ]:
ANIO_INI_CAL = 2021
ANIO_FIN_CAL = 2025

CODIGO_EJEMPLO = "086060003000000010230000000000"      # escribe aquí el CODIGO elegido en la sección 9, entre comillas
NDVI_DESNUDO = 0.3         # SUPUESTO: por debajo de este NDVI mensual se marca "posible campo desnudo o estrés"

assert CODIGO_EJEMPLO is not None, 'Primero elige un CODIGO en la sección 9 y escríbelo arriba'
fila_ej = gdf[gdf['CODIGO'] == str(CODIGO_EJEMPLO).strip()]
assert len(fila_ej) > 0, 'Ese CODIGO no existe en gdf'

import json
geom = ee.Geometry(json.loads(fila_ej.to_crs(4326).iloc[[0]].to_json())['features'][0]['geometry'])


def con_ndvi(img):
    """NDVI Sentinel-2 con máscara SCL."""
    m = enmascarar_s2(img)
    nir, red = m.select("B8"), m.select("B4")
    return nir.subtract(red).divide(nir.add(red)).rename("NDVI")


def ndvi_predio_mes(anio, mes):
    ini = ee.Date.fromYMD(anio, mes, 1)
    fin = ini.advance(1, 'month')
    img = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(geom).filterDate(ini, fin)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
        .map(con_ndvi).median()
    )
    vals = img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=geom, scale=10, maxPixels=1e9
    )
    return ee.Feature(None, vals).set('anio', anio).set('mes', mes)


filas_p = []
for anio in range(ANIO_INI_CAL, ANIO_FIN_CAL + 1):
    fc_p = ee.FeatureCollection(
        ee.List.sequence(1, 12).map(lambda m: ndvi_predio_mes(anio, m))
    )
    filas_p += [x['properties'] for x in fc_p.getInfo()['features']]
    print(f'  año {anio} listo')

df_p = pd.DataFrame(filas_p)
if 'NDVI' not in df_p.columns:
    df_p['NDVI'] = float('nan')
df_p = df_p.sort_values(['anio', 'mes']).reset_index(drop=True)
curva = df_p.groupby('mes')['NDVI'].mean().round(3)
df_p.to_csv(f'{RUTA_BASE}/ndvi_predio_ejemplo_serie.csv', index=False)
curva.to_csv(f'{RUTA_BASE}/ndvi_predio_ejemplo.csv')
print(curva)

fig, ax = plt.subplots(figsize=(9, 4))
for anio_g, g in df_p.groupby('anio'):
    ax.plot(g['mes'], g['NDVI'], color='lightgray', linewidth=1)
ax.plot(curva.index, curva.values, color='forestgreen', marker='o', linewidth=2, label='NDVI promedio')
ax.axhline(NDVI_DESNUDO, color='firebrick', linestyle='--', label=f'umbral supuesto ({NDVI_DESNUDO})')
ax.set_xticks(range(1, 13))
ax.set_xlabel('Mes del año')
ax.set_ylabel('NDVI del predio')
ax.legend(loc='upper left')
if 'clima' in globals():
    ax_ll = ax.twinx()
    ax_ll.bar(clima.index, clima['lluvia_mm'], color='dodgerblue', alpha=0.25)
    ax_ll.set_ylabel('Lluvia de la zona (mm)')
plt.title(f'Predio {CODIGO_EJEMPLO}: verdor por mes (líneas grises = cada año)')
plt.tight_layout()
plt.savefig(f'{RUTA_BASE}/ndvi_predio_ejemplo.png', dpi=200, bbox_inches='tight')
plt.show()

meses_bajos = [int(m) for m, v in curva.items() if v < NDVI_DESNUDO]
print(f'Meses con NDVI promedio por debajo de {NDVI_DESNUDO}: {meses_bajos if meses_bajos else "ninguno"}')

## Riesgo por cobertura y uso agropecuario

In [ ]:
# ============================================================
# RELACION ENTRE NIVEL DE RIESGO Y COBERTURA DEL SUELO
# ============================================================

# gdf ya contiene 1.848 predios únicos (sección 2) con cobertura (9b) y riesgo (6)
predios_unicos = gdf.copy()
assert len(predios_unicos) == 1848 and 'cobertura_dominante' in predios_unicos.columns, 'Corre antes las secciones 6 y 9b'

tabla_riesgo_cobertura = pd.crosstab(
    predios_unicos["cobertura_dominante"],
    predios_unicos["Nivel_Riesgo"],
    margins=True
)

print("PARCELAS POR COBERTURA Y NIVEL DE RIESGO")
display(tabla_riesgo_cobertura)

print("\nDISTRIBUCION DEL RIESGO ALTO")
riesgo_alto = predios_unicos[
    predios_unicos["Nivel_Riesgo"] == "Alto"
]

display(
    riesgo_alto["cobertura_dominante"]
    .value_counts()
    .to_frame("parcelas")
)

print(
    "\nPorcentaje de parcelas de riesgo alto:",
    round(len(riesgo_alto) / len(predios_unicos) * 100, 2),
    "%"
)


print("\nNIVEL DE RIESGO SEGUN USO AGROPECUARIO (WorldCover >= 50 % cultivo+pastizal)")
display(pd.crosstab(predios_unicos["uso_agropecuario"], predios_unicos["Nivel_Riesgo"], margins=True))

print("\nRIESGO ALTO: predios pequeños (< 0,1 ha) y fuera de uso agropecuario")
print("  < 0,1 ha:", int((riesgo_alto["area_ha"] < 0.1).sum()), "de", len(riesgo_alto))
print("  no agropecuarios:", int((~riesgo_alto["uso_agropecuario"].astype(bool)).sum()), "de", len(riesgo_alto))
print("No interpretar estos predios como estrés de un cultivo concreto: cobertura modelada (2021), píxeles de 10 m y lluvia de 5-11 km.")

## 11. Exportar archivos para la página de demo

Guarda en Drive una tabla con todos los predios (riesgo, coordenadas, área, distancia al distrito) y un archivo con los parámetros y fechas usados, para poder reproducir el resultado.
Necesita haber corrido las secciones 0 a 6 y la 9.

In [ ]:
import json

assert 'en_distrito' in gdf.columns, 'Corre primero la sección 9'

centros = gdf.geometry.centroid.to_crs(4326)
cols_demo = [
    'CODIGO', 'area_ha', 'en_distrito', 'dist_distrito_km', 'NDVI', 'NDMI', 'SAVI', 'V_Vulnerability',
    'precip_mm', 'precip_normal_mm', 'Precip_pct_normal', 'Amenaza_Norm', 'Riesgo_Index', 'Nivel_Riesgo',
]
cols_demo += [c for c in ['cobertura_dominante', 'pct_agropecuario', 'uso_agropecuario', 'soil_clay_pct'] if c in gdf.columns]
exp = gdf[cols_demo].copy()
exp['baja_confianza'] = exp['area_ha'] < 0.1   # predios menores a 0,1 ha: el píxel de 10 m es grande frente a ellos
exp['lat'] = centros.y.round(5).values
exp['lon'] = centros.x.round(5).values
exp.round(4).to_csv(f'{RUTA_BASE}/predios_para_demo_chirps_{f(FIN)}.csv', index=False)

params = {
    'fecha_fin_analisis': f(FIN),
    'ventana_sentinel2_dias': VENTANA_DIAS,
    'fuente_lluvia': CHIRPS_ID,
    'resolucion_lluvia_m': ESCALA_CHIRPS,
    'periodo_lluvia': [f(LLUVIA_INI), f(LLUVIA_FIN)],
    'dias_periodo_lluvia': dias_periodo,
    'n_pentadas': N_PENTADAS,
    'anios_normal': ANIOS_USADOS,
    'nubosidad_max_pct': NUBOSIDAD_MAX,
    'escala_s2_m': ESCALA_S2,
    'anios_para_lluvia_normal': ANIOS_CLIMA,
    'lluvia_actual_zona_mm': round(float(LLUVIA_ACTUAL_REGIONAL_MM), 2),
    'lluvia_normal_zona_mm': round(float(LLUVIA_NORMAL_REGIONAL_MM), 2),
    'amenaza': 'clip(1 - lluvia_actual/lluvia_normal_del_mismo_pixel, 0, 1)',
    'pesos_V': PESOS_V,
    'umbrales_indices': {k: list(v) for k, v in UMBRALES.items()},
    'peso_amenaza': PESO_AMENAZA,
    'cortes_nivel_riesgo': {'Critico': 0.75, 'Alto': 0.50, 'Moderado': 0.25},
    'conjuntos_de_datos': {
        'sentinel2': 'COPERNICUS/S2_SR_HARMONIZED',
        'lluvia': CHIRPS_ID,
        'lluvia_comparacion': 'NASA/GPM_L3/IMERG_V07 (notebook 2)',
        'suelo': 'ISRIC SoilGrids (clay_mean)',
    },
    'distrito_de_riego': {
        'registro': 'Registro de distritos de riego activos (ADR), archivo descargado el 14-11-2026: ASUREP, 3.613 ha, 416 familias',
        'poligono': 'Servicio de Distritos de Adecuación de Tierras (UPRA y ANT), cruzado con los predios en QGIS',
        'regla_de_asignacion': 'predios que intersecan el polígono (por verificar: puede incluir predios que solo tocan el borde)',
        'predios_dentro': int(gdf['en_distrito'].sum()),
        'punto_referencia_adr': {'lat': round(LAT_DISTRITO, 5), 'lon': round(LON_DISTRITO, 5)},
    },
    'n_predios': int(len(gdf)),
    'predios_por_nivel': {k: int(v) for k, v in gdf['Nivel_Riesgo'].value_counts().items()},
    'advertencia': 'Umbrales y pesos son supuestos iniciales sin calibrar; resultados sin validar en campo. La fuente de lluvia cambia sustancialmente el diagnóstico (ver comparacion_gpm_vs_chirps).',
}
with open(f'{RUTA_BASE}/parametros_modelo_chirps_{f(FIN)}.json', 'w', encoding='utf-8') as fh:
    json.dump(params, fh, ensure_ascii=False, indent=2)

print(f'✅ Guardados predios_para_demo_chirps_{f(FIN)}.csv ({len(exp)} filas) y parametros_modelo_chirps_{f(FIN)}.json en Drive')

## Fin del notebook 1

Archivos de salida en Drive: `predios_para_demo_chirps_2026-11-15.csv`, `parametros_modelo_chirps_2026-11-15.json` y los CSV auxiliares. Para comparaciones y yuca, abre el notebook 2; **no necesita las variables de memoria de este notebook**.
